In [ ]:
import os

# used for configuring biogeme use of GPU, unused
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
# os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Destination-choice + stay/move model via Larch

In [ ]:
import os
import sys
from datetime import UTC, datetime

import larch as lx
from larch import PX

sys.path.insert(0, os.path.abspath(".."))
from lib import io as lio
from lib import util as lut


In [ ]:
year = 2018
num_alternatives = 50
unixtime = int(datetime.now(UTC).timestamp())

### Read data

In [ ]:
df_train = lio.read_estdata(
    year=year,
    num_alternatives=num_alternatives,
)
print(df_train.shape)

### Reshape to long format, build the Larch dataset

`lib.util.build_long_data` builds the long `(person_id, alt)` table shared by this notebook and
`modeling_torch_choice.ipynb`: `alt=0` is staying, `alt=1..num_alternatives` are the move alternatives, with the
stay/move-context values for shared coefficients (e.g. `proportion_same_age_18_34`) written under the
same column name so they tie to one coefficient downstream, and `log_pop_offset` (destination
population on move rows, origin population on the stay row) left as an un-parameterized term. See its
docstring for the full column-by-column breakdown, including the two `modeling_mnl.ipynb` race terms
omitted for the same reasons noted in this notebook's intro cell.

The returned `long_df` is already sorted by `(person_id, alt)`; `Dataset.construct.from_idca` takes it
directly (indexed by `(caseid, altid)`) -- no manual reshape into arrays needed, unlike the torch-choice
port.


In [ ]:
long_df, STAY_ONLY_TERMS, SHARED_TERMS, MOVE_ONLY_TERMS = lut.build_long_data(
    df_train, num_alternatives
)
varnames = STAY_ONLY_TERMS + SHARED_TERMS + MOVE_ONLY_TERMS

num_persons = df_train["person_id"].nunique()
num_alts = num_alternatives + 1  # 51: alt=0 (stay) + alt=1..50 (move)

idca = long_df.set_index(["person_id", "alt"])[["choice", "log_pop_offset"] + varnames]
ds = lx.Dataset.construct.from_idca(idca, crack=True)
ds


### Setting up the model

One `P(name) * X(name)` term per `varnames` entry via the `PX` shorthand, summed on a plain local
variable and assigned to `m.utility_ca` once at the end -- **not** built with `m.utility_ca += ...` in
a loop, which silently discards everything but the last term (see intro cell). No separate ASC/intercept
term is added -- `stay` in `STAY_ONLY_TERMS` is already an explicit ASC for staying, matching
`fit_intercept=False` in the torch-choice port / Biogeme not adding an implicit ASC of its own.

`log_pop_offset` gets a coefficient too, then `m.lock_value("log_pop_offset", 1)` pins it at exactly 1
(`holdfast`), reproducing Biogeme's bare `log(Variable(...))` calls (and xlogit's `addit=`) without
needing a custom subclass the way the torch-choice port did.


In [ ]:
m = lx.Model(ds)
m.title = f"us_mnl_{year}_{unixtime}"
m.compute_engine = "numba"

# all alternatives have the same utility function
# stay-specific columns have their values zeroed out for destination alternatives and vice versa
# PX represents a column multiplied by a coefficient that will be estimated
total_utility = PX(varnames[0])
for name in varnames[1:]:
    total_utility = total_utility + PX(name)
total_utility = total_utility + PX("log_pop_offset")
m.utility_ca = total_utility

m.choice_ca_var = "choice"
# all alternatives are available for everyone (matches av[i] = 1 for all i in modeling_mnl.ipynb);
# no availability_ca_var needed.

# fix the size term coefficient, it is a constant
m.lock_value("log_pop_offset", 1)

m.ordering = [
    ("Stay", "stay.*"),
    ("Shared", "proportion.*|median_.*|unemp_rate|vacancy_rate"),
    ("Destination-only", "destchoice.*"),
    ("Offset", "log_pop_offset"),
]


In [ ]:
# optional cell: turns on the nested structure

# nested logit: alt=0 (stay) stays a direct root child (== a degenerate nest fixed at 1.0);
# alts 1..num_alternatives go under a "Move" nest with an estimated logsum coefficient.
m.graph.new_node(
    parameter="mu_move",
    children=list(range(1, num_alternatives + 1)),
    name="Move",
)
m.set_value("mu_move", value=0.5, initvalue=0.5, minimum=0.001, maximum=1.0)

m.ordering = [
    ("Stay", "stay.*"),
    ("Shared", "proportion.*|median_.*|unemp_rate|vacancy_rate"),
    ("Destination-only", "destchoice.*"),
    ("Nesting", "mu_.*"),
    ("Offset", "log_pop_offset"),
]
m.title = f"us_nested_{year}_{unixtime}"


### Fitting

In [ ]:
print("null log-likelihood:", m.loglike())


In [ ]:
result = m.maximize_loglike(method="BHHH")
result


In [ ]:
m.calculate_parameter_covariance()
m.parameter_summary()


In [ ]:
report = lx.Reporter(title=m.title)
report << "# Parameter Summary" << m.parameter_summary()
report << "# Estimation Statistics" << m.estimation_statistics()
report.save(
    f"results/{m.title}.html",
    overwrite=True,
    metadata=m.dumps(),
)
m.save(f"results/{m.title}_spec.yaml")